# Hyperparameter grid

In [ ]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    from google.colab import drive

    drive.mount("/content/drive")

import torch

from config import RESULTS_DIR, SHAH_PLM, SHAH_SEEDS
from experiments.grid import grid_search, winners

GRID_OUT = RESULTS_DIR / "grid.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("grid ->", GRID_OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

## Six encoders, 3 learning rates x 3 batch sizes

In [ ]:
# validation only -- no test_df reaches finetune, so selection never sees test.
# grid_search scans grid.csv, skips configs already there, and appends one row per
# config straight to Drive, so a disconnect costs one run and nothing is clobbered.
grid_search(GRID_OUT, models=list(SHAH_PLM), seeds=SHAH_SEEDS, device=DEVICE)

## Winners

In [ ]:
import polars as pl

print(winners(pl.read_csv(GRID_OUT)))